# Step 2 — Deduplicate workflow history

> Keep the latest valid workflow state per `uid` so downstream steps see one row per unique workflow (not per commit in its history).

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/a-dataset-of-github-actions-workflow-histories-workflows.csv` | CSV (~1.5 GB — workflow history of 49,510 repositories) | External: Cardoen et al. (2024) workflow-histories dataset |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/2_deduplicated_workflows.csv` | CSV (one row per workflow) | (legacy/standalone — Step 3 re-derives the snapshot from the raw CSV; not consumed downstream) |
| `data/step2_report.json` | JSON | (legacy/standalone — not consumed downstream) |



In [3]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 2 — starting: deduplicate workflows (heaviest read; ~1.5 GB CSV, may take a few minutes)...", flush=True)
"""Step 2: deduplicate workflows dataset.

Reads the raw workflows CSV (3.4M rows), keeps only the most recent entry per
uid, filters out deleted workflows (git_change_type=D) and invalid workflows
(valid_workflow=False), and outputs a deduplicated dataset.
"""

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import polars as pl


@dataclass
class DedupStats:
    total_rows: int = 0
    rows_written: int = 0
    removed_invalid_workflow: int = 0
    removed_deleted: int = 0
    removed_duplicates: int = 0

def deduplicate_workflows(input_path: Path, output_path: Path) -> DedupStats:
    stats = DedupStats()

    from dataset_utils.integrity import hash_source_file
    input_sha = hash_source_file(input_path, "step 2 input")

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")
    stats.total_rows = len(df)

    stats.removed_invalid_workflow = df.filter(pl.col("valid_workflow") != True).height
    df = df.filter(pl.col("valid_workflow") == True)

    stats.removed_deleted = df.filter(pl.col("git_change_type") == "D").height
    df = df.filter(pl.col("git_change_type") != "D")

    original_count = len(df)
    # Deterministic tiebreaker so unique(keep="first") is reproducible.
    df = df.sort(
        ["committed_date", "uid", "file_path"],
        descending=[True, False, False],
    )
    df = df.unique(subset=["uid"], keep="first", maintain_order=True)
    stats.removed_duplicates = original_count - len(df)

    stats.rows_written = len(df)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_sha256": input_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path = output_path.parent / "step2_report.json"
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Deduplicate workflows dataset, keeping the most recent entry per uid."
    )
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/a-dataset-of-github-actions-workflow-histories-workflows.csv"),
        help="Input workflows CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/2_deduplicated_workflows.csv"),
        help="Output deduplicated workflows CSV file path.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = deduplicate_workflows(args.input, args.output)

    print("Workflow deduplication finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print("-" * 40)
    print(f"Total rows read: {stats.total_rows}")
    print(f"Rows written: {stats.rows_written}")
    print("-" * 40)
    print(f"Removed invalid workflows: {stats.removed_invalid_workflow}")
    print(f"Removed deleted workflows: {stats.removed_deleted}")
    print(f"Removed duplicate entries: {stats.removed_duplicates}")

main()


Step 2 — starting: deduplicate workflows (heaviest read; ~1.5 GB CSV, may take a few minutes)...
  reading input CSV (may take a moment for large files)...
  writing output...
Workflow deduplication finished.
Input: data/a-dataset-of-github-actions-workflow-histories-workflows.csv
Output: data/2_deduplicated_workflows.csv
----------------------------------------
Total rows read: 3418911
Rows written: 260444
----------------------------------------
Removed invalid workflows: 181367
Removed deleted workflows: 70358
Removed duplicate entries: 2906742


---

**Next step:** [`03_merge_datasets.ipynb`](./03_merge_datasets.ipynb)
